In [ ]:
import os, shutil, subprocess, sys, zipfile, json, time, re, statistics
from pathlib import Path

inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))
candidates = [p for p in inputs.rglob("src/train/trainer.py") if "nanowm-code" in str(p)]
if not candidates:
    raise RuntimeError(f"no nanowm-code tree under {inputs}")
mounted = sorted(candidates, key=lambda q: len(q.parts))[0].parent.parent.parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
assert (root / "configs" / "m4_main.yaml").exists(), sorted(p.name for p in root.iterdir())
assert (root / "scripts" / "run_m4_eval.py").exists()
assert "ThroughputCollapse" in (root / "scripts" / "train.py").read_text(), "stale code dataset"
print("project root:", root)

subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "lpips"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

matches = [p for p in inputs.rglob("manifest.json") if "nanowm-m3-latents" in str(p)]
if len(matches) != 1:
    raise RuntimeError(f"expected exactly one precomputed m3 manifest, got {matches}")
DATA_DIR = matches[0].parent
print("data dir:", DATA_DIR, json.load(open(matches[0]))["num_trajectories"], "trajectories")

def run_train(cfg_path):
    """train.py with live output AND a captured copy, so steps/s lines can
    be parsed afterwards. Exit codes: 0 done, 2 ticket deadline, 3
    throughput collapse (both checkpointed), anything else = crash."""
    proc = subprocess.Popen([sys.executable, "scripts/train.py", "--config", str(cfg_path)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    proc.wait()
    return proc.returncode, lines

def steps_per_second(lines, skip=2):
    rates = [float(m.group(1)) for m in (re.search(r"([0-9.]+) steps/s", l) for l in lines) if m]
    if len(rates) <= skip:
        raise RuntimeError(f"too few throughput samples: {rates}")
    return statistics.median(rates[skip:])   # first intervals include compile


In [ ]:
# Diagnosis (~5 min GPU): the exact M4 config, 300 steps, checkpoint saves
# at 100/200, watchdog OFF, once under compile "reduce-overhead" (what part
# 1 ran) and once under "default". train.py prints steps/s and GPU memory
# every 25 steps, so a collapse right after the step-100 save shows up
# directly. The "default" run doubles as the throughput calibration for
# the main run. Tickets: 2 x 0.05 GPU-h.
import yaml
TICKET_HOURS = 2.0
CALIB_TICKET_HOURS = 0.05
SAFETY = 0.9

base = yaml.safe_load(Path("configs/m4_main.yaml").read_text())
base["data"]["kwargs"]["data_dir"] = str(DATA_DIR)

def diag_run(mode):
    d = json.loads(json.dumps(base))
    d["run"].update(run_id=f"m4_diag_{mode}", ticket_hours=CALIB_TICKET_HOURS, log_interval=25,
                    expected_steps_per_second=None,
                    purpose=f"Diagnose part-1 collapse: 300 steps, checkpoints at 100/200, compile_mode={mode}")
    d["trainer"].update(total_steps=300, warmup_steps=30, log_interval=25, compile_mode=mode)
    d["checkpoint"].update(dir=f"/kaggle/working/runs/m4_diag_{mode}/checkpoints", interval_steps=100)
    path = Path(f"configs/m4_diag_{mode}.yaml"); path.write_text(yaml.safe_dump(d))
    t0 = time.time(); code, lines = run_train(path); secs = time.time() - t0
    rows = [(int(m.group(1)), float(m.group(2)), float(m.group(3)))
            for m in (re.search(r"step ([0-9]+) .* ([0-9.]+) steps/s gpu_used ([0-9.]+)GiB", l) for l in lines) if m]
    n_ckpt = len(list(Path(d["checkpoint"]["dir"]).glob("*.pt")))
    shutil.rmtree(f"/kaggle/working/runs/m4_diag_{mode}", ignore_errors=True)
    return {"mode": mode, "exit_code": code, "seconds": secs, "checkpoints": n_ckpt,
            "rows": rows, "steps_per_second": steps_per_second(lines)}

diag = {}
for mode in ("reduce-overhead", "default"):
    diag[mode] = diag_run(mode)
    print()
    print(f"=== {mode}: exit {diag[mode]['exit_code']}, {diag[mode]['seconds']:.0f} s, "
          f"{diag[mode]['checkpoints']} checkpoints, median {diag[mode]['steps_per_second']:.2f} steps/s ===")
    for step, sps, gpu in diag[mode]["rows"]:
        print(f"   step {step:4d}  {sps:6.2f} steps/s  gpu_used {gpu:5.2f} GiB")
Path("/kaggle/working/m4_diagnosis.json").write_text(json.dumps(diag, indent=2))

d_ok = diag["default"]
if d_ok["exit_code"] != 0 or d_ok["checkpoints"] < 3:
    raise RuntimeError("the default-mode diagnosis itself failed -- do not start the main run")
sps = d_ok["steps_per_second"]
plan = {"steps_per_second": sps, "ticket_hours": TICKET_HOURS, "safety": SAFETY,
        "total_steps": int(SAFETY * sps * TICKET_HOURS * 3600), "checkpoint_interval": int(sps * 1800)}
Path("/kaggle/working/m4_plan.json").write_text(json.dumps(plan, indent=2))
print(json.dumps(plan, indent=2))


In [ ]:
# Main run. Ticket approved by the user (see TICKET_HOURS above). Values
# go only into this Kaggle copy of the config; the committed one stays
# null (tests/test_m4_eval.py).
cfg = json.loads(json.dumps(base))
cfg["run"].update(ticket_hours=TICKET_HOURS, expected_steps_per_second=plan["steps_per_second"])
cfg["trainer"].update(total_steps=plan["total_steps"])
cfg["checkpoint"].update(interval_steps=plan["checkpoint_interval"])
cfg_path = Path("configs/m4_main.yaml"); cfg_path.write_text(yaml.safe_dump(cfg))
print(cfg_path.read_text())

code, lines = run_train(cfg_path)
print(f"train.py exit code: {code}  (0 done, 2 ticket deadline, 3 throughput collapse)")
print(Path("budget/ledger.jsonl").read_text().strip().splitlines()[-1])
if code not in (0, 2, 3):
    raise RuntimeError("training crashed")


In [ ]:
# Eval on whatever checkpoint exists (also after exit 2/3). Held-out
# scenes (seeds 1000-1003) are rendered here: inference-side, weekly
# quota, not the training budget.
ckpt_dir = Path(cfg["checkpoint"]["dir"])
assert (ckpt_dir / "latest.json").exists(), list(ckpt_dir.iterdir())
subprocess.run([sys.executable, "scripts/preprocess/precompute_dataset.py",
                "--tier", "heldout", "--out-dir", "/kaggle/working/nanowm_data/heldout",
                "--device", "cuda"], check=True)
subprocess.run([sys.executable, "scripts/run_m4_eval.py",
                "--checkpoint-dir", str(ckpt_dir),
                "--heldout-dir", "/kaggle/working/nanowm_data/heldout",
                "--train-dir", str(DATA_DIR),
                "--preset", "40m", "--out-dir", "/kaggle/working/m4_eval"], check=True)
print(json.dumps({k: v for k, v in json.load(open("/kaggle/working/m4_eval/m4_eval.json")).items()
                  if k not in ("heldout", "train_reference")}, indent=2))


In [ ]:
# Keep only the checkpoint latest.json points at; drop the project copy.
latest = json.loads((ckpt_dir / "latest.json").read_text())
for f in ckpt_dir.glob("*.pt"):
    if f.name != latest["path"]:
        f.unlink()
print("kept:", latest, [f.name for f in ckpt_dir.iterdir()])
shutil.copy("budget/ledger.jsonl", "/kaggle/working/ledger.jsonl")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/project", ignore_errors=True)
shutil.rmtree("/kaggle/working/nanowm_data", ignore_errors=True)
